# FFT-Like Transformations for Image Processing (Java + ImageJ)

Java-only notebook for an ImageJ/FIBA pipeline with three FFT-like local frequency transforms: **Gabor Transform**, **Chirplet Transform**, and **Morlet Wavelet**.

## Main story flow
1. Preflight Java/Maven + project checks
2. STORM-like preprocessing
3. Run Java tile-montage pipeline (crop / polar / SOL / mask / reconstruction)
4. Generate transform images
5. Compare transform tile stacks + mask performance
6. Show inline comparison figures and references

Supplementary diagnostics are moved to an **Appendix** section near the end.

## Transform sections and biological intuition

### Gabor Transform
Gabor uses a Gaussian-windowed sinusoid, giving localized joint sensitivity to orientation and spatial frequency.

$$g(x,y)=\exp\!\left(-\frac{x_\theta^2+\gamma^2 y_\theta^2}{2\sigma^2}\right)\cos\!\left(\frac{2\pi x_\theta}{\lambda}+\psi\right)$$

Use here: strong baseline for oriented fibrillar texture (collagen-like bundles).

### Chirplet Transform
Chirplet extends Gabor by adding local frequency sweep (chirp), improving flexibility when local periodicity changes across space.

$$c(x,y)\propto \exp\!\left(-\frac{x_\theta^2+\gamma^2 y_\theta^2}{2\sigma^2}\right)\cos\!\left(2\pi\left(\frac{x_\theta}{\lambda}+\kappa x_\theta^2\right)\right)$$

Use here: useful when texture spacing bends or drifts within tile neighborhoods.

### Morlet Wavelet
Morlet is closely related to complex Gabor wavelets; in this notebook we treat it as a wavelet-style complex oriented filter bank.

Use here: multiscale/oriented wavelet perspective for comparison to Chirplet and Gabor with the same downstream mask-based metrics.

## Deeper math: wavelets, multi-resolution analysis, and FFT-like local transforms

### 1) Multi-resolution analysis (MRA) foundation
Classical wavelet analysis is built from nested approximation spaces

$$\cdots \subset V_{-1} \subset V_0 \subset V_1 \subset \cdots$$

with detail spaces $W_j$ such that

$$V_{j+1}=V_j \oplus W_j.$$

A scaling function $\phi$ spans coarse structure, while a wavelet $\psi$ spans detail. In 2D images, separable filter banks produce subbands (LL, LH, HL, HH) across scales.

### 2) Filter-bank view (discrete wavelet transform)
At each level, low-pass/high-pass analysis filters with downsampling give

$$a_{j+1}[k]=\sum_n h[n-2k]a_j[n],\qquad d_{j+1}[k]=\sum_n g[n-2k]a_j[n].$$

For images, this is applied along rows/columns. This is the rigorous multiscale decomposition view behind “wavelet-like” analysis.

### 3) Where Morlet/Gabor/Chirplet fit
This notebook uses localized FFT-like atoms rather than a full orthogonal DWT tree:

- **Morlet wavelet**: complex sinusoid under a Gaussian envelope (continuous-wavelet flavor).
- **Gabor transform**: mathematically very close to Morlet in practice, emphasizing local orientation/frequency energy.
- **Chirplet transform**: adds a chirp term (local frequency sweep), useful when local spacing varies within a tile.

### 4) Why this is still multi-scale in practice
Even without a strict dyadic pyramid, the bank is multiscale/multi-orientation because responses are aggregated over orientations (and, in diagnostic views, multiple wavelengths). That provides a practical localized time-frequency/space-frequency representation for textured biological structure.

### 5) Link to the scoring used here
For each transform map $T$, this notebook scores

$$\mathrm{score}=0.7\big(\mathbb{E}[T\mid M>t]-\mathbb{E}[T\mid M\le t]\big)+0.3\,\mathrm{corr}(T,R),$$

where $M$ is the mask and $R$ is reconstruction. So the notebook optimizes both segmentation contrast and reconstruction consistency, not just transform energy alone.

## References for biological and imaging applications

### Gabor / Morlet-style localized wavelets in imaging
- Daugman, J. G. (1985). *Uncertainty relation for resolution in space, spatial frequency, and orientation optimized by two-dimensional visual cortical filters*. **JOSA A**.
- Manjunath, B. S., & Ma, W.-Y. (1996). *Texture features for browsing and retrieval of image data*. **IEEE TPAMI**.
- Jain, A. K., Prabhakar, S., Hong, L., & Pankanti, S. (2000). *Filterbank-based fingerprint matching*. **IEEE TIP**.
- Unser, M. (1995). *Texture classification and segmentation using wavelet frames*. **IEEE TIP**.

### Chirplet / chirp-based time-frequency methods in bio-imaging contexts
- Mann, S., & Haykin, S. (1991). *The chirplet transform: Physical considerations*. **IEEE Transactions on Signal Processing**.
- O’Neill, J. C., et al. (2010). Chirplet/time-frequency approaches for nonstationary biomedical signal characterization (ECG/EEG examples). **Biomedical Signal Processing literature**.
- Li, C., Durand, L. G., & Blaber, A. P. (1997). *Time–frequency distributions for biomedical signals* (comparative discussion including chirp-sensitive methods). **Medical & Biological Engineering & Computing**.

### Biological orientation-structure quantification examples
- Rezakhaniha, R., et al. (2012). *Experimental investigation of collagen waviness and orientation in the arterial adventitia using confocal laser scanning microscopy*. **Biomechanics and Modeling in Mechanobiology**.
- Bredfeldt, J. S., et al. (2014). *Computational segmentation of collagen fibers from second-harmonic generation images of breast cancer*. **Journal of Biomedical Optics**.

These references support why localized, orientation-sensitive, and multiscale/time-frequency transforms are appropriate for structured biological texture analysis.

In [1]:
System.out.println("Java kernel is running ✅");
System.out.println("java.version: " + System.getProperty("java.version"));
System.out.println("java.vendor:  " + System.getProperty("java.vendor"));
System.out.println("java.home:    " + System.getProperty("java.home"));
System.out.println("user.dir:     " + System.getProperty("user.dir"));

Java kernel is running ✅
java.version: 25.0.2
java.vendor:  Eclipse Adoptium
java.home:    C:\Users\dunnmk\AppData\Local\Programs\Eclipse Adoptium\jdk-25.0.2.10-hotspot
user.dir:     c:\Users\dunnmk\repos\imgjplugin\notebooks


In [3]:
import java.io.*;
import java.nio.charset.StandardCharsets;
import java.nio.file.*;
import java.util.*;
import java.util.regex.*;

Path findProjectRoot(Path start) {
    Path p = start.toAbsolutePath().normalize();
    for (int i = 0; i < 12 && p != null; i++) {
        if (Files.exists(p.resolve("FFT").resolve("pom.xml"))) return p;
        p = p.getParent();
    }
    return null;
}

Path findOnPath(String exe) {
    String path = System.getenv("PATH");
    if (path == null || path.isBlank()) return null;
    String[] exts = System.getProperty("os.name").toLowerCase().contains("win")
        ? new String[] {"", ".cmd", ".bat", ".exe"}
        : new String[] {""};

    for (String part : path.split(Pattern.quote(File.pathSeparator))) {
        if (part == null || part.isBlank()) continue;
        Path dir = Paths.get(part.trim());
        if (!Files.isDirectory(dir)) continue;
        for (String ext : exts) {
            Path cand = dir.resolve(exe + ext);
            if (Files.isRegularFile(cand)) return cand;
        }
    }
    return null;
}

Path findMavenExecutable() {
    Path p = findOnPath("mvn");
    if (p != null) return p;
    String mh = System.getenv("MAVEN_HOME");
    if (mh != null && !mh.isBlank()) {
        Path bin = Paths.get(mh, "bin");
        for (String c : new String[] {"mvn.cmd", "mvn.bat", "mvn.exe", "mvn"}) {
            Path m = bin.resolve(c);
            if (Files.isRegularFile(m)) return m;
        }
    }

    List<Path> bins = Arrays.asList(
        Paths.get(System.getProperty("user.home"), "tools", "maven", "apache-maven-3.9.6", "bin"),
        Paths.get(System.getProperty("user.home"), "tools", "apache-maven-3.9.6", "bin"),
        Paths.get(System.getProperty("user.home"), ".tools", "apache-maven-3.9.6", "bin")
    );
    for (Path bin : bins) {
        for (String c : new String[] {"mvn.cmd", "mvn.bat", "mvn.exe", "mvn"}) {
            Path m = bin.resolve(c);
            if (Files.isRegularFile(m)) return m;
        }
    }
    return null;
}

String runAndCollect(List<String> cmd, Path cwd, int timeoutSeconds, int maxLines) throws Exception {
    ProcessBuilder pb = new ProcessBuilder(cmd);
    if (cwd != null) pb.directory(cwd.toFile());
    pb.redirectErrorStream(true);
    Process proc = pb.start();

    StringBuilder sb = new StringBuilder();
    int shown = 0;
    long deadline = System.currentTimeMillis() + timeoutSeconds * 1000L;

    try (BufferedReader br = new BufferedReader(new InputStreamReader(proc.getInputStream(), StandardCharsets.UTF_8))) {
        String line;
        while (true) {
            while (br.ready() && (line = br.readLine()) != null) {
                if (shown < maxLines) {
                    sb.append(line).append("\n");
                    shown++;
                }
            }
            if (!proc.isAlive()) break;
            if (System.currentTimeMillis() > deadline) {
                proc.destroyForcibly();
                throw new RuntimeException("Timed out: " + String.join(" ", cmd));
            }
            Thread.sleep(100);
        }
        while ((line = br.readLine()) != null) {
            if (shown < maxLines) {
                sb.append(line).append("\n");
                shown++;
            }
        }
    }

    int exit = proc.waitFor();
    sb.append("[exit=").append(exit).append("]\n");
    if (exit != 0) throw new RuntimeException("Command failed: " + String.join(" ", cmd) + "\n" + sb);
    return sb.toString();
}

Path root = findProjectRoot(Paths.get(System.getProperty("user.dir")));
if (root == null) throw new RuntimeException("Could not find project root containing FFT/pom.xml");
Path fftDir = root.resolve("FFT");
Path pom = fftDir.resolve("pom.xml");

Path javaExe = findOnPath("java");
Path javacExe = findOnPath("javac");
Path mvnExe = findMavenExecutable();

System.out.println("Project root: " + root);
System.out.println("FFT dir:      " + fftDir);
System.out.println("pom.xml:      " + pom + " (exists=" + Files.exists(pom) + ")");
System.out.println();
System.out.println("java  -> " + (javaExe == null ? "MISSING" : javaExe));
System.out.println("javac -> " + (javacExe == null ? "MISSING" : javacExe));
System.out.println("mvn   -> " + (mvnExe == null ? "MISSING" : mvnExe));

if (javaExe == null || javacExe == null || mvnExe == null) {
    throw new RuntimeException("Preflight failed: java/javac/mvn must all be available on PATH.");
}

System.out.println();
System.out.println(runAndCollect(Arrays.asList(javaExe.toString(), "-version"), fftDir, 30, 40));
System.out.println(runAndCollect(Arrays.asList(javacExe.toString(), "-version"), fftDir, 30, 20));
System.out.println(runAndCollect(Arrays.asList(mvnExe.toString(), "-v"), fftDir, 45, 60));
System.out.println("Preflight complete ✅");

Project root: c:\Users\dunnmk\repos\imgjplugin
FFT dir:      c:\Users\dunnmk\repos\imgjplugin\FFT
pom.xml:      c:\Users\dunnmk\repos\imgjplugin\FFT\pom.xml (exists=true)

java  -> C:\Users\dunnmk\AppData\Local\Programs\Eclipse Adoptium\jdk-25.0.2.10-hotspot\bin\java.exe
javac -> C:\Users\dunnmk\AppData\Local\Programs\Eclipse Adoptium\jdk-25.0.2.10-hotspot\bin\javac.exe
mvn   -> C:\Users\dunnmk\tools\maven\apache-maven-3.9.6\bin\mvn.cmd

openjdk version "25.0.2" 2026-01-20 LTS
OpenJDK Runtime Environment Temurin-25.0.2+10 (build 25.0.2+10-LTS)
OpenJDK 64-Bit Server VM Temurin-25.0.2+10 (build 25.0.2+10-LTS, mixed mode, sharing)
[exit=0]

javac 25.0.2
[exit=0]


Apache Maven 3.9.6 (bc0240f3c744dd6b6ec2920b3cd08dcc295161ae)
Maven home: C:\Users\dunnmk\tools\maven\apache-maven-3.9.6
Java version: 25.0.2, vendor: Eclipse Adoptium, runtime: C:\Users\dunnmk\AppData\Local\Programs\Eclipse Adoptium\jdk-25.0.2.10-hotspot
Default locale: en_US, platform encoding: UTF-8
OS name: "windows 11", ver

In [4]:
import java.awt.image.BufferedImage;
import javax.imageio.ImageIO;

final double SIGMA_SMALL = 1.0;
final double SIGMA_LARGE = 3.0;
final double THRESH_STD = 0.50;
final double GAMMA = 0.70;

double[][] toGray(BufferedImage img) {
    int h = img.getHeight();
    int w = img.getWidth();
    double[][] g = new double[h][w];
    for (int y = 0; y < h; y++) {
        for (int x = 0; x < w; x++) {
            int rgb = img.getRGB(x, y);
            int r = (rgb >> 16) & 0xff;
            int gg = (rgb >> 8) & 0xff;
            int b = rgb & 0xff;
            g[y][x] = 0.299 * r + 0.587 * gg + 0.114 * b;
        }
    }
    return g;
}

BufferedImage fromGray(double[][] g) {
    int h = g.length;
    int w = g[0].length;
    BufferedImage out = new BufferedImage(w, h, BufferedImage.TYPE_BYTE_GRAY);
    for (int y = 0; y < h; y++) {
        for (int x = 0; x < w; x++) {
            int v = (int)Math.round(Math.max(0, Math.min(255, g[y][x])));
            int rgb = (v << 16) | (v << 8) | v;
            out.setRGB(x, y, rgb);
        }
    }
    return out;
}

double[][] copy2D(double[][] a) {
    int h = a.length, w = a[0].length;
    double[][] b = new double[h][w];
    for (int y = 0; y < h; y++) System.arraycopy(a[y], 0, b[y], 0, w);
    return b;
}

void normalize01(double[][] a) {
    double min = Double.POSITIVE_INFINITY, max = Double.NEGATIVE_INFINITY;
    for (double[] row : a) for (double v : row) {
        min = Math.min(min, v);
        max = Math.max(max, v);
    }
    double span = Math.max(1e-9, max - min);
    for (int y = 0; y < a.length; y++) {
        for (int x = 0; x < a[0].length; x++) {
            a[y][x] = (a[y][x] - min) / span;
        }
    }
}

double[] gaussianKernel1D(double sigma) {
    int radius = Math.max(1, (int)Math.ceil(3.0 * sigma));
    int n = radius * 2 + 1;
    double[] k = new double[n];
    double sum = 0.0;
    for (int i = -radius; i <= radius; i++) {
        double v = Math.exp(-(i * i) / (2.0 * sigma * sigma));
        k[i + radius] = v;
        sum += v;
    }
    for (int i = 0; i < n; i++) k[i] /= sum;
    return k;
}

double[][] convolveHorizontal(double[][] src, double[] k) {
    int h = src.length, w = src[0].length;
    int r = k.length / 2;
    double[][] out = new double[h][w];
    for (int y = 0; y < h; y++) {
        for (int x = 0; x < w; x++) {
            double s = 0.0;
            for (int i = -r; i <= r; i++) {
                int xx = Math.min(w - 1, Math.max(0, x + i));
                s += src[y][xx] * k[i + r];
            }
            out[y][x] = s;
        }
    }
    return out;
}

double[][] convolveVertical(double[][] src, double[] k) {
    int h = src.length, w = src[0].length;
    int r = k.length / 2;
    double[][] out = new double[h][w];
    for (int y = 0; y < h; y++) {
        for (int x = 0; x < w; x++) {
            double s = 0.0;
            for (int i = -r; i <= r; i++) {
                int yy = Math.min(h - 1, Math.max(0, y + i));
                s += src[yy][x] * k[i + r];
            }
            out[y][x] = s;
        }
    }
    return out;
}

double[][] gaussianBlur(double[][] src, double sigma) {
    double[] k = gaussianKernel1D(sigma);
    return convolveVertical(convolveHorizontal(src, k), k);
}

double[][] stormLikeEnhance(double[][] gray255) {
    double[][] norm = copy2D(gray255);
    normalize01(norm);

    double[][] gSmall = gaussianBlur(norm, SIGMA_SMALL);
    double[][] gLarge = gaussianBlur(norm, SIGMA_LARGE);

    int h = norm.length, w = norm[0].length;
    double[][] dog = new double[h][w];
    double mean = 0.0, sq = 0.0;
    int n = h * w;

    for (int y = 0; y < h; y++) {
        for (int x = 0; x < w; x++) {
            dog[y][x] = gSmall[y][x] - gLarge[y][x];
            mean += dog[y][x];
            sq += dog[y][x] * dog[y][x];
        }
    }
    mean /= n;
    double std = Math.sqrt(Math.max(0.0, (sq / n) - mean * mean));
    double thresh = mean + THRESH_STD * std;

    double[][] out = new double[h][w];
    for (int y = 0; y < h; y++) {
        for (int x = 0; x < w; x++) {
            double v = Math.max(0.0, dog[y][x] - thresh);
            out[y][x] = Math.pow(v, GAMMA);
        }
    }

    double min = Double.POSITIVE_INFINITY, max = Double.NEGATIVE_INFINITY;
    for (double[] row : out) for (double v : row) {
        min = Math.min(min, v);
        max = Math.max(max, v);
    }
    double span = Math.max(1e-9, max - min);
    for (int y = 0; y < h; y++) {
        for (int x = 0; x < w; x++) {
            out[y][x] = 255.0 * (out[y][x] - min) / span;
        }
    }
    return out;
}

Path rootPre = findProjectRoot(Paths.get(System.getProperty("user.dir")));
Path preDir = rootPre.resolve("notebooks").resolve("_assets").resolve("fiba_wavelet_montage").resolve("preprocessed");
Files.createDirectories(preDir);

List<Path> rawInputs = Arrays.asList(
    Paths.get("C:\\Users\\dunnmk\\Downloads\\C15D5P001 (1).jpg"),
    Paths.get("C:\\Users\\dunnmk\\OneDrive - Michigan Medicine\\Pictures\\Picture1.jpg")
);
List<String> rawBases = Arrays.asList("C15D5P001_1", "Picture1");

for (int i = 0; i < rawInputs.size(); i++) {
    Path in = rawInputs.get(i);
    if (!Files.isRegularFile(in)) throw new RuntimeException("Missing input: " + in);
    BufferedImage img = ImageIO.read(in.toFile());
    if (img == null) throw new RuntimeException("Could not read image: " + in);

    double[][] gray = toGray(img);
    double[][] enhanced = stormLikeEnhance(gray);
    Path out = preDir.resolve(rawBases.get(i) + "_clean_storm.jpg");
    ImageIO.write(fromGray(enhanced), "jpg", out.toFile());
    System.out.println("Preprocessed image written: " + out);
}

System.out.println("STORM-like preprocessing complete ✅");

Preprocessed image written: c:\Users\dunnmk\repos\imgjplugin\notebooks\_assets\fiba_wavelet_montage\preprocessed\C15D5P001_1_clean_storm.jpg
Preprocessed image written: c:\Users\dunnmk\repos\imgjplugin\notebooks\_assets\fiba_wavelet_montage\preprocessed\Picture1_clean_storm.jpg
STORM-like preprocessing complete ✅


In [5]:
import java.util.regex.Matcher;

Path rootRun = findProjectRoot(Paths.get(System.getProperty("user.dir")));
Path fftDirRun = rootRun.resolve("FFT");
Path mvnRun = findMavenExecutable();
if (mvnRun == null) throw new RuntimeException("Maven executable not found");

Path preDirRun = rootRun.resolve("notebooks").resolve("_assets").resolve("fiba_wavelet_montage").resolve("preprocessed");
List<Path> inputImages = Arrays.asList(
    Paths.get("C:\\Users\\dunnmk\\Downloads\\C15D5P001 (1).jpg"),
    Paths.get("C:\\Users\\dunnmk\\OneDrive - Michigan Medicine\\Pictures\\Picture1.jpg")
);
List<String> baseNames = Arrays.asList("C15D5P001_1", "Picture1");
List<Path> outputDirs = Arrays.asList(
    rootRun.resolve("notebooks").resolve("_assets").resolve("fiba_wavelet_montage").resolve("generated_from_source"),
    rootRun.resolve("notebooks").resolve("_assets").resolve("fiba_wavelet_montage").resolve("generated_from_source_picture1")
);

for (int idx = 0; idx < inputImages.size(); idx++) {
    Path rawImage = inputImages.get(idx);
    String base = baseNames.get(idx);
    Path outDir = outputDirs.get(idx);
    Path cleaned = preDirRun.resolve(base + "_clean_storm.jpg");

    if (!Files.isRegularFile(cleaned)) throw new RuntimeException("Missing preprocessed image: " + cleaned);
    Files.createDirectories(outDir);

    List<String> cmd = Arrays.asList(
        mvnRun.toString(),
        "-B",
        "-Dtest=fftanalysis.imagej.GenerateTileMontageFromSourceTest",
        "-Dfiba.input=" + cleaned.toString(),
        "-Dfiba.output=" + outDir.toString(),
        "-Dfiba.base=" + base,
        "-Dfiba.tilesY=10",
        "test"
    );

    System.out.println("\nRunning pipeline for " + base + " using " + cleaned);
    System.out.println(runAndCollect(cmd, fftDirRun, 900, 250));

    Path boxes = outDir.resolve(base + "_tile_boxes.jpg");
    Path stack = outDir.resolve(base + "_tile_montage.jpg");
    Path csv = outDir.resolve(base + "_tile_results.csv");
    if (!Files.isRegularFile(boxes) || !Files.isRegularFile(stack) || !Files.isRegularFile(csv)) {
        throw new RuntimeException("Missing expected pipeline outputs in: " + outDir);
    }

    Pattern tilePattern = Pattern.compile("^" + Pattern.quote(base) + "_tile(\\d+)_crop\\.jpg$");
    List<Integer> tileIds = new ArrayList<>();
    try (DirectoryStream<Path> ds = Files.newDirectoryStream(outDir, "*_crop.jpg")) {
        for (Path p : ds) {
            Matcher m = tilePattern.matcher(p.getFileName().toString());
            if (m.matches()) tileIds.add(Integer.parseInt(m.group(1)));
        }
    }
    tileIds.sort(Comparator.naturalOrder());
    if (tileIds.isEmpty()) throw new RuntimeException("No crop tiles found in " + outDir);
    System.out.println("Generated tile ids: " + tileIds);
}

System.out.println("\nDual-image pipeline generation complete ✅");


Running pipeline for C15D5P001_1 using c:\Users\dunnmk\repos\imgjplugin\notebooks\_assets\fiba_wavelet_montage\preprocessed\C15D5P001_1_clean_storm.jpg

[INFO] Scanning for projects...
[INFO] 
[INFO] ---------------------< com.mikdunn:imgjplugin-fft >---------------------
[INFO] Building ImageJ FFT Orientation Plugin 0.1.0-SNAPSHOT
[INFO]   from pom.xml
[INFO] --------------------------------[ jar ]---------------------------------
[INFO] 
[INFO] --- enforcer:3.6.1:enforce (enforce-maven) @ imgjplugin-fft ---
[INFO] Rule 0: org.apache.maven.enforcer.rules.version.RequireMavenVersion passed
[INFO] 
[INFO] --- resources:3.3.1:resources (default-resources) @ imgjplugin-fft ---
[INFO] Copying 1 resource from resources to target\classes
[INFO] 
[INFO] --- compiler:3.15.0:compile (default-compile) @ imgjplugin-fft ---
[INFO] Recompiling the module because of added or removed source files.
[INFO] Compiling 7 source files with javac [debug target 1.8] to target\classes
[WARNING] bootstrap cla

In [20]:
import java.awt.image.BufferedImage;
import javax.imageio.ImageIO;
import java.util.*;
import java.util.regex.*;
import java.nio.file.*;

// -------------------------------
// Analysis configuration (Gabor only)
// -------------------------------
final int GABOR_ORIENTATIONS = 8;
final int GABOR_KERNEL_SIZE = 13;
final double GABOR_SIGMA = 2.2;
final double GABOR_LAMBDA = 5.5;
final double GABOR_GAMMA = 0.65;

double[][] imageToGrayArray(BufferedImage img) {
    int h = img.getHeight(), w = img.getWidth();
    double[][] a = new double[h][w];
    for (int y = 0; y < h; y++) {
        for (int x = 0; x < w; x++) {
            int rgb = img.getRGB(x, y);
            int r = (rgb >> 16) & 0xff;
            int g = (rgb >> 8) & 0xff;
            int b = rgb & 0xff;
            a[y][x] = 0.299 * r + 0.587 * g + 0.114 * b;
        }
    }
    return a;
}

double[][] gaborKernel(int size, double sigma, double theta, double lambda, double gamma, double psi) {
    int r = size / 2;
    double[][] k = new double[size][size];
    for (int y = -r; y <= r; y++) {
        for (int x = -r; x <= r; x++) {
            double xr = x * Math.cos(theta) + y * Math.sin(theta);
            double yr = -x * Math.sin(theta) + y * Math.cos(theta);
            double gauss = Math.exp(-(xr * xr + (gamma * gamma) * yr * yr) / (2.0 * sigma * sigma));
            double wave = Math.cos(2.0 * Math.PI * xr / lambda + psi);
            k[y + r][x + r] = gauss * wave;
        }
    }
    return k;
}

double[][] convolveSame(double[][] src, double[][] kernel) {
    int h = src.length, w = src[0].length;
    int kh = kernel.length, kw = kernel[0].length;
    int ry = kh / 2, rx = kw / 2;
    double[][] out = new double[h][w];
    for (int y = 0; y < h; y++) {
        for (int x = 0; x < w; x++) {
            double s = 0.0;
            for (int j = -ry; j <= ry; j++) {
                int yy = Math.max(0, Math.min(h - 1, y + j));
                for (int i = -rx; i <= rx; i++) {
                    int xx = Math.max(0, Math.min(w - 1, x + i));
                    s += src[yy][xx] * kernel[j + ry][i + rx];
                }
            }
            out[y][x] = s;
        }
    }
    return out;
}

double[][] gaborEnergyMap(double[][] src, int orientations, int kSize, double sigma, double lambda, double gamma) {
    int h = src.length, w = src[0].length;
    double[][] out = new double[h][w];
    int nOri = Math.max(1, orientations);

    for (int oi = 0; oi < nOri; oi++) {
        double theta = (Math.PI * oi) / nOri;
        double[][] kRe = gaborKernel(kSize, sigma, theta, lambda, gamma, 0.0);
        double[][] kIm = gaborKernel(kSize, sigma, theta, lambda, gamma, Math.PI / 2.0);
        double[][] re = convolveSame(src, kRe);
        double[][] im = convolveSame(src, kIm);
        for (int y = 0; y < h; y++) {
            for (int x = 0; x < w; x++) {
                double mag = Math.sqrt(re[y][x] * re[y][x] + im[y][x] * im[y][x]);
                if (mag > out[y][x]) out[y][x] = mag;
            }
        }
    }
    return out;
}

BufferedImage toGrayImageScaled(double[][] a) {
    int h = a.length, w = a[0].length;
    double min = Double.POSITIVE_INFINITY, max = Double.NEGATIVE_INFINITY;
    for (double[] row : a) for (double v : row) {
        min = Math.min(min, v);
        max = Math.max(max, v);
    }
    double span = Math.max(1e-9, max - min);
    BufferedImage out = new BufferedImage(w, h, BufferedImage.TYPE_BYTE_GRAY);
    for (int y = 0; y < h; y++) {
        for (int x = 0; x < w; x++) {
            int v = (int)Math.round(255.0 * (a[y][x] - min) / span);
            v = Math.max(0, Math.min(255, v));
            int rgb = (v << 16) | (v << 8) | v;
            out.setRGB(x, y, rgb);
        }
    }
    return out;
}

Path rootW = findProjectRoot(Paths.get(System.getProperty("user.dir")));
List<String> basesW = Arrays.asList("C15D5P001_1", "Picture1");
List<Path> dirsW = Arrays.asList(
    rootW.resolve("notebooks").resolve("_assets").resolve("fiba_wavelet_montage").resolve("generated_from_source"),
    rootW.resolve("notebooks").resolve("_assets").resolve("fiba_wavelet_montage").resolve("generated_from_source_picture1")
);

for (int i = 0; i < basesW.size(); i++) {
    String base = basesW.get(i);
    Path dir = dirsW.get(i);

    Pattern p = Pattern.compile("^" + Pattern.quote(base) + "_tile(\\d+)_crop\\.jpg$");
    List<Integer> ids = new ArrayList<>();
    try (DirectoryStream<Path> ds = Files.newDirectoryStream(dir, "*_crop.jpg")) {
        for (Path f : ds) {
            Matcher m = p.matcher(f.getFileName().toString());
            if (m.matches()) ids.add(Integer.parseInt(m.group(1)));
        }
    }
    ids.sort(Comparator.naturalOrder());
    if (ids.isEmpty()) throw new RuntimeException("No crop tiles for transform generation in " + dir);

    for (int id : ids) {
        Path cropPath = dir.resolve(base + "_tile" + id + "_crop.jpg");
        BufferedImage crop = ImageIO.read(cropPath.toFile());
        if (crop == null) throw new RuntimeException("Could not read crop tile: " + cropPath);

        double[][] gray = imageToGrayArray(crop);
        double[][] transform = gaborEnergyMap(gray, GABOR_ORIENTATIONS, GABOR_KERNEL_SIZE, GABOR_SIGMA, GABOR_LAMBDA, GABOR_GAMMA);

        Path gaborOut = dir.resolve(base + "_tile" + id + "_gabor.jpg");
        ImageIO.write(toGrayImageScaled(transform), "jpg", gaborOut.toFile());
    }

    System.out.println("Transform images generated for " + base + " tiles: " + ids);
}

System.out.println("Transform stage complete ✅ mode=gabor, model=Gabor-o8-k13");

Transform images generated for C15D5P001_1 tiles: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10]
Transform images generated for Picture1 tiles: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10]
Transform stage complete ✅ mode=gabor, model=Gabor-o8-k13


In [21]:
import java.awt.Color;
import java.awt.Font;
import java.awt.Graphics2D;
import java.awt.RenderingHints;
import java.awt.geom.Rectangle2D;
import java.awt.image.BufferedImage;
import javax.imageio.ImageIO;
import java.io.IOException;
import java.nio.file.*;
import java.util.ArrayList;
import java.util.Arrays;
import java.util.Comparator;
import java.util.List;
import java.util.regex.Matcher;
import java.util.regex.Pattern;

BufferedImage readOrPlaceholder(Path p, int w, int h, String label) throws IOException {
    if (Files.exists(p)) {
        BufferedImage img = ImageIO.read(p.toFile());
        if (img != null) return img;
    }
    BufferedImage miss = new BufferedImage(w, h, BufferedImage.TYPE_INT_RGB);
    Graphics2D g = miss.createGraphics();
    g.setColor(new Color(245, 245, 245));
    g.fillRect(0, 0, w, h);
    g.setColor(new Color(200, 60, 60));
    g.setFont(new Font("SansSerif", Font.BOLD, 18));
    g.drawString("MISSING", 16, 30);
    g.setColor(Color.DARK_GRAY);
    g.setFont(new Font("SansSerif", Font.PLAIN, 13));
    g.drawString(label, 16, 54);
    g.dispose();
    return miss;
}

void drawFit(Graphics2D g, BufferedImage src, int x, int y, int w, int h) {
    double sx = w / (double) src.getWidth();
    double sy = h / (double) src.getHeight();
    double s = Math.min(sx, sy);
    int nw = Math.max(1, (int) Math.round(src.getWidth() * s));
    int nh = Math.max(1, (int) Math.round(src.getHeight() * s));
    int ox = x + (w - nw) / 2;
    int oy = y + (h - nh) / 2;
    g.drawImage(src, ox, oy, nw, nh, null);
}

int drawSectionTitle(Graphics2D g, String title, int x, int y, int w) {
    g.setColor(new Color(24, 44, 74));
    g.setFont(new Font("SansSerif", Font.BOLD, 22));
    g.drawString(title, x, y + 22);
    g.setColor(new Color(220, 226, 235));
    g.fill(new Rectangle2D.Double(x, y + 28, w, 2));
    return y + 36;
}

int drawSingleImageSection(Graphics2D g, String title, Path p, int x, int y, int w, int h) throws IOException {
    int yy = drawSectionTitle(g, title, x, y, w);
    g.setColor(new Color(232, 232, 232));
    g.fillRect(x - 1, yy - 1, w + 2, h + 2);
    BufferedImage img = readOrPlaceholder(p, w, h, p.getFileName().toString());
    drawFit(g, img, x, yy, w, h);
    return yy + h + 18;
}

int drawGridSection(Graphics2D g, String title, List<Integer> ids, String key, Path assetsDir, String baseName, int x, int y, int columns, int cellW, int cellH, int gap) throws IOException {
    int yy = drawSectionTitle(g, title, x, y, columns * cellW + (columns - 1) * gap);
    int rows = (int) Math.ceil(ids.size() / (double) columns);
    g.setFont(new Font("SansSerif", Font.PLAIN, 13));

    for (int i = 0; i < ids.size(); i++) {
        int id = ids.get(i);
        int r = i / columns;
        int c = i % columns;
        int cx = x + c * (cellW + gap);
        int cy = yy + r * (cellH + 26 + gap);

        Path p = assetsDir.resolve(baseName + "_tile" + id + "_" + key + ".jpg");
        BufferedImage img = readOrPlaceholder(p, cellW, cellH, p.getFileName().toString());

        g.setColor(new Color(30, 30, 30));
        g.drawString("Tile " + id, cx + 4, cy + 14);
        g.setColor(new Color(232, 232, 232));
        g.fillRect(cx - 1, cy + 17 - 1, cellW + 2, cellH + 2);
        drawFit(g, img, cx, cy + 17, cellW, cellH);
    }

    return yy + rows * (cellH + 26 + gap) + 8;
}

Path rootFig = findProjectRoot(Paths.get(System.getProperty("user.dir")));
List<Path> inputImages2 = Arrays.asList(
    Paths.get("C:\\Users\\dunnmk\\Downloads\\C15D5P001 (1).jpg"),
    Paths.get("C:\\Users\\dunnmk\\OneDrive - Michigan Medicine\\Pictures\\Picture1.jpg")
);
List<String> baseNames2 = Arrays.asList("C15D5P001_1", "Picture1");
List<Path> outputDirs2 = Arrays.asList(
    rootFig.resolve("notebooks").resolve("_assets").resolve("fiba_wavelet_montage").resolve("generated_from_source"),
    rootFig.resolve("notebooks").resolve("_assets").resolve("fiba_wavelet_montage").resolve("generated_from_source_picture1")
);

for (int caseIdx = 0; caseIdx < baseNames2.size(); caseIdx++) {
    String base = baseNames2.get(caseIdx);
    Path assetsDir = outputDirs2.get(caseIdx);
    Path originalPath = inputImages2.get(caseIdx);

    Path tileStackPath = assetsDir.resolve(base + "_tile_montage.jpg");
    if (!Files.isRegularFile(originalPath)) throw new RuntimeException("Missing source image: " + originalPath);
    if (!Files.isRegularFile(tileStackPath)) throw new RuntimeException("Missing tile stack: " + tileStackPath);

    Pattern tilePattern = Pattern.compile("^" + Pattern.quote(base) + "_tile(\\d+)_crop\\.jpg$");
    List<Integer> tileIds = new ArrayList<>();
    try (DirectoryStream<Path> ds = Files.newDirectoryStream(assetsDir, "*_crop.jpg")) {
        for (Path p : ds) {
            Matcher m = tilePattern.matcher(p.getFileName().toString());
            if (m.matches()) tileIds.add(Integer.parseInt(m.group(1)));
        }
    }
    tileIds.sort(Comparator.naturalOrder());
    if (tileIds.isEmpty()) throw new RuntimeException("No generated tile images found in " + assetsDir);

    List<Integer> firstTen = tileIds.stream().filter(i -> i >= 1 && i <= 10).toList();
    if (firstTen.isEmpty()) firstTen = tileIds;

    int margin = 28;
    int pageW = 1320;
    int usableW = pageW - 2 * margin;
    int columns = 5;
    int gap = 14;
    int cellW = (usableW - (columns - 1) * gap) / columns;
    int cellH = 165;
    int rows = (int) Math.ceil(firstTen.size() / (double) columns);
    int gridH = rows * (cellH + 26 + gap) + 8;

    int hOriginal = 380;
    int hTileStack = 330;
    int titleTop = 62;
    int bottomPad = 28;
    int sectionGap = 8;

    int pageH = titleTop
        + (36 + hOriginal + 18)
        + sectionGap
        + (36 + hTileStack + 18)
        + sectionGap
        + (36 + gridH)
        + sectionGap
        + (36 + gridH)
        + sectionGap
        + (36 + gridH)
        + sectionGap
        + (36 + gridH)
        + sectionGap
        + (36 + gridH)
        + sectionGap
        + (36 + gridH)
        + bottomPad;

    BufferedImage canvas = new BufferedImage(pageW, pageH, BufferedImage.TYPE_INT_RGB);
    Graphics2D g = canvas.createGraphics();
    g.setRenderingHint(RenderingHints.KEY_ANTIALIASING, RenderingHints.VALUE_ANTIALIAS_ON);
    g.setRenderingHint(RenderingHints.KEY_INTERPOLATION, RenderingHints.VALUE_INTERPOLATION_BILINEAR);
    g.setColor(Color.WHITE);
    g.fillRect(0, 0, pageW, pageH);

    g.setColor(new Color(12, 33, 64));
    g.setFont(new Font("SansSerif", Font.BOLD, 30));
    g.drawString("Final FIBA Gabor Figure (Generated from Source): " + base, margin, 38);
    g.setFont(new Font("SansSerif", Font.PLAIN, 14));
    g.drawString("Order: source image -> tile stack -> images 1-10 -> transform -> polar -> SOL -> mask -> reconstruction", margin, 56);

    int y = titleTop;
    y = drawSingleImageSection(g, "1) Source image", originalPath, margin, y, usableW, hOriginal);
    y += sectionGap;
    y = drawSingleImageSection(g, "2) Tiles stack (generated by pipeline)", tileStackPath, margin, y, usableW, hTileStack);
    y += sectionGap;
    y = drawGridSection(g, "3) Images numbered 1-10", firstTen, "crop", assetsDir, base, margin, y, columns, cellW, cellH, gap);
    y += sectionGap;
    y = drawGridSection(g, "4) Gabor transform image (orientation-bank energy)", firstTen, "gabor", assetsDir, base, margin, y, columns, cellW, cellH, gap);
    y += sectionGap;
    y = drawGridSection(g, "5) Polar coordinates", firstTen, "polar", assetsDir, base, margin, y, columns, cellW, cellH, gap);
    y += sectionGap;
    y = drawGridSection(g, "6) SOL graph", firstTen, "sol", assetsDir, base, margin, y, columns, cellW, cellH, gap);
    y += sectionGap;
    y = drawGridSection(g, "7) Mask", firstTen, "mask", assetsDir, base, margin, y, columns, cellW, cellH, gap);
    y += sectionGap;
    y = drawGridSection(g, "8) Reconstruction", firstTen, "rec", assetsDir, base, margin, y, columns, cellW, cellH, gap);

    g.dispose();

    Path outPath = assetsDir.resolve(base + "_final_figure_java_gabor.png");
    ImageIO.write(canvas, "png", outPath.toFile());
    System.out.println("Final gabor figure written: " + outPath);
}

System.out.println("Dual final-figure generation complete ✅");

Final gabor figure written: c:\Users\dunnmk\repos\imgjplugin\notebooks\_assets\fiba_wavelet_montage\generated_from_source\C15D5P001_1_final_figure_java_gabor.png
Final gabor figure written: c:\Users\dunnmk\repos\imgjplugin\notebooks\_assets\fiba_wavelet_montage\generated_from_source_picture1\Picture1_final_figure_java_gabor.png
Dual final-figure generation complete ✅


## Appendix — supplementary deep-dive analyses

The cells below are intentionally kept as supplementary material so the main notebook narrative stays concise.

Included appendix content:
- extended transform scoring diagnostics
- Gabor filter-bank visualization details
- dominant-orientation histogram analysis
- additional side-by-side stack diagnostics
- figure path troubleshooting helper

## Model scoring formula and what it means biologically

The scoring cell combines two objectives per tile:

1. **Mask separation** (inside vs. outside response contrast),
2. **Correlation with reconstruction** (agreement with the downstream reconstruction image).

Let $T$ be normalized transform intensity, $M$ the mask, and $R$ the normalized reconstruction. Then

$$\mathrm{sep}=\mathbb{E}[T\mid M>t]-\mathbb{E}[T\mid M\le t]$$

$$\mathrm{corr}=\mathrm{Pearson}(T,R)$$

and the notebook score is

$$\mathrm{score}=0.7\,\mathrm{sep}+0.3\,\mathrm{corr}$$

Biological reading: larger separation means the transform emphasizes biologically relevant segmented structure; larger correlation means consistency with the pipeline’s reconstruction behavior. Together, this balances specificity and coherence.

Reference context for multiscale/energy formulations in imaging:
- Unser, M. (1995). **Texture classification and segmentation using wavelet frames**. *IEEE TIP*.
- Portilla, J., & Simoncelli, E. P. (2000). **A parametric texture model based on joint statistics of complex wavelet coefficients**. *IJCV*.

In [18]:
import java.awt.image.BufferedImage;
import javax.imageio.ImageIO;
import java.nio.file.*;
import java.util.*;
import java.util.regex.*;

record ModelCfg(String label, String mode, String family, int levels, int gaborOri, int gaborK, double gaborSigma, double gaborLambda, double gaborGamma) {}

record ModelScore(String label, double sepMean, double corrMean, double score, int nTiles) {}

double[][] grayFromImage(BufferedImage img) {
    int h = img.getHeight(), w = img.getWidth();
    double[][] out = new double[h][w];
    for (int y = 0; y < h; y++) {
        for (int x = 0; x < w; x++) {
            int rgb = img.getRGB(x, y);
            int r = (rgb >> 16) & 0xff;
            int g = (rgb >> 8) & 0xff;
            int b = rgb & 0xff;
            out[y][x] = 0.299 * r + 0.587 * g + 0.114 * b;
        }
    }
    return out;
}

double[][] normalize01(double[][] a) {
    int h = a.length, w = a[0].length;
    double min = Double.POSITIVE_INFINITY, max = Double.NEGATIVE_INFINITY;
    for (double[] row : a) for (double v : row) {
        min = Math.min(min, v);
        max = Math.max(max, v);
    }
    double span = Math.max(1e-9, max - min);
    double[][] out = new double[h][w];
    for (int y = 0; y < h; y++) {
        for (int x = 0; x < w; x++) out[y][x] = (a[y][x] - min) / span;
    }
    return out;
}

double pearson(double[][] a, double[][] b) {
    int h = Math.min(a.length, b.length);
    int w = Math.min(a[0].length, b[0].length);
    double sa = 0, sb = 0;
    int n = h * w;
    for (int y = 0; y < h; y++) {
        for (int x = 0; x < w; x++) {
            sa += a[y][x];
            sb += b[y][x];
        }
    }
    double ma = sa / n, mb = sb / n;
    double num = 0, da = 0, db = 0;
    for (int y = 0; y < h; y++) {
        for (int x = 0; x < w; x++) {
            double xa = a[y][x] - ma;
            double xb = b[y][x] - mb;
            num += xa * xb;
            da += xa * xa;
            db += xb * xb;
        }
    }
    double den = Math.sqrt(Math.max(1e-12, da * db));
    return num / den;
}

double[] inOutMeans(double[][] map01, double[][] maskGray) {
    int h = Math.min(map01.length, maskGray.length);
    int w = Math.min(map01[0].length, maskGray[0].length);
    double in = 0, out = 0;
    int nin = 0, nout = 0;
    for (int y = 0; y < h; y++) {
        for (int x = 0; x < w; x++) {
            boolean inside = maskGray[y][x] > 32.0;
            if (inside) {
                in += map01[y][x];
                nin++;
            } else {
                out += map01[y][x];
                nout++;
            }
        }
    }
    double minIn = nin == 0 ? 0.0 : in / nin;
    double minOut = nout == 0 ? 0.0 : out / nout;
    return new double[] { minIn, minOut };
}

double[][] transformForModel(double[][] gray, ModelCfg m) {
    return gaborEnergyMap(gray, m.gaborOri(), m.gaborK(), m.gaborSigma(), m.gaborLambda(), m.gaborGamma());
}

Path rootCmp = findProjectRoot(Paths.get(System.getProperty("user.dir")));
List<String> basesCmp = Arrays.asList("C15D5P001_1", "Picture1");
List<Path> dirsCmp = Arrays.asList(
    rootCmp.resolve("notebooks").resolve("_assets").resolve("fiba_wavelet_montage").resolve("generated_from_source"),
    rootCmp.resolve("notebooks").resolve("_assets").resolve("fiba_wavelet_montage").resolve("generated_from_source_picture1")
);

List<ModelCfg> models = Arrays.asList(
    new ModelCfg("Gabor-o8-k13", "gabor", "na", 0, 8, 13, 2.2, 5.5, 0.65)
);

List<ModelScore> scores = new ArrayList<>();

for (ModelCfg m : models) {
    double sepSum = 0.0;
    double corrSum = 0.0;
    int count = 0;

    for (int ds = 0; ds < basesCmp.size(); ds++) {
        String base = basesCmp.get(ds);
        Path dir = dirsCmp.get(ds);

        Pattern p = Pattern.compile("^" + Pattern.quote(base) + "_tile(\\d+)_crop\\.jpg$");
        List<Integer> ids = new ArrayList<>();
        try (DirectoryStream<Path> stream = Files.newDirectoryStream(dir, "*_crop.jpg")) {
            for (Path f : stream) {
                Matcher mm = p.matcher(f.getFileName().toString());
                if (mm.matches()) ids.add(Integer.parseInt(mm.group(1)));
            }
        }
        ids.sort(Comparator.naturalOrder());

        for (int id : ids) {
            Path cropPath = dir.resolve(base + "_tile" + id + "_crop.jpg");
            Path maskPath = dir.resolve(base + "_tile" + id + "_mask.jpg");
            Path recPath = dir.resolve(base + "_tile" + id + "_rec.jpg");
            if (!Files.isRegularFile(cropPath) || !Files.isRegularFile(maskPath) || !Files.isRegularFile(recPath)) continue;

            BufferedImage crop = ImageIO.read(cropPath.toFile());
            BufferedImage mask = ImageIO.read(maskPath.toFile());
            BufferedImage rec = ImageIO.read(recPath.toFile());
            if (crop == null || mask == null || rec == null) continue;

            double[][] gray = grayFromImage(crop);
            double[][] tf = normalize01(transformForModel(gray, m));
            double[][] maskGray = grayFromImage(mask);
            double[][] recGray = normalize01(grayFromImage(rec));

            double[] means = inOutMeans(tf, maskGray);
            double sep = means[0] - means[1];
            double corr = pearson(tf, recGray);

            sepSum += sep;
            corrSum += corr;
            count++;
        }
    }

    if (count > 0) {
        double sepMean = sepSum / count;
        double corrMean = corrSum / count;
        double score = 0.7 * sepMean + 0.3 * corrMean;
        scores.add(new ModelScore(m.label(), sepMean, corrMean, score, count));
    }
}

scores.sort((a, b) -> Double.compare(b.score(), a.score()));

System.out.println("Gabor-only score across both datasets (higher is better)");
System.out.println("score = 0.7*(mask-separation) + 0.3*(corr-with-rec)");
System.out.println("label\tsepMean\tcorrMean\tscore\tnTiles");
for (ModelScore s : scores) {
    System.out.printf(Locale.US, "%s\t%.5f\t%.5f\t%.5f\t%d%n", s.label(), s.sepMean(), s.corrMean(), s.score(), s.nTiles());
}

if (!scores.isEmpty()) {
    ModelScore best = scores.get(0);
    System.out.printf(Locale.US, "\nBest fit model: %s (score=%.5f)%n", best.label(), best.score());
}


Gabor-only score across both datasets (higher is better)
score = 0.7*(mask-separation) + 0.3*(corr-with-rec)
label	sepMean	corrMean	score	nTiles
Gabor-o8-k13	0.04454	0.73174	0.25070	20

Best fit model: Gabor-o8-k13 (score=0.25070)


In [12]:
import java.awt.Color;
import java.awt.Font;
import java.awt.Graphics2D;
import java.awt.image.BufferedImage;
import javax.imageio.ImageIO;
import java.nio.file.*;
import java.util.*;

// Gabor filter-bank visualization settings
final int VIEW_TILE_ID = 1;
final int VIEW_ORIENTATIONS = 8;
final int VIEW_KERNEL_SIZE = 13;
final double VIEW_SIGMA = 2.2;
final double VIEW_GAMMA = 0.65;
final double[] VIEW_LAMBDAS = new double[] {3.5, 5.5, 8.0};

double[][] gray2D(BufferedImage img) {
    int h = img.getHeight(), w = img.getWidth();
    double[][] g = new double[h][w];
    for (int y = 0; y < h; y++) {
        for (int x = 0; x < w; x++) {
            int rgb = img.getRGB(x, y);
            int r = (rgb >> 16) & 0xff;
            int gg = (rgb >> 8) & 0xff;
            int b = rgb & 0xff;
            g[y][x] = 0.299 * r + 0.587 * gg + 0.114 * b;
        }
    }
    return g;
}

double[][] gaborKernelView(int size, double sigma, double theta, double lambda, double gamma, double psi) {
    int r = size / 2;
    double[][] k = new double[size][size];
    for (int y = -r; y <= r; y++) {
        for (int x = -r; x <= r; x++) {
            double xr = x * Math.cos(theta) + y * Math.sin(theta);
            double yr = -x * Math.sin(theta) + y * Math.cos(theta);
            double gauss = Math.exp(-(xr * xr + (gamma * gamma) * yr * yr) / (2.0 * sigma * sigma));
            double wave = Math.cos(2.0 * Math.PI * xr / lambda + psi);
            k[y + r][x + r] = gauss * wave;
        }
    }
    return k;
}

double[][] convolveSameView(double[][] src, double[][] kernel) {
    int h = src.length, w = src[0].length;
    int kh = kernel.length, kw = kernel[0].length;
    int ry = kh / 2, rx = kw / 2;
    double[][] out = new double[h][w];
    for (int y = 0; y < h; y++) {
        for (int x = 0; x < w; x++) {
            double s = 0.0;
            for (int j = -ry; j <= ry; j++) {
                int yy = Math.max(0, Math.min(h - 1, y + j));
                for (int i = -rx; i <= rx; i++) {
                    int xx = Math.max(0, Math.min(w - 1, x + i));
                    s += src[yy][xx] * kernel[j + ry][i + rx];
                }
            }
            out[y][x] = s;
        }
    }
    return out;
}

BufferedImage toGray(double[][] a) {
    int h = a.length, w = a[0].length;
    double min = Double.POSITIVE_INFINITY, max = Double.NEGATIVE_INFINITY;
    for (double[] row : a) for (double v : row) {
        min = Math.min(min, v);
        max = Math.max(max, v);
    }
    double span = Math.max(1e-9, max - min);
    BufferedImage out = new BufferedImage(w, h, BufferedImage.TYPE_BYTE_GRAY);
    for (int y = 0; y < h; y++) {
        for (int x = 0; x < w; x++) {
            int v = (int)Math.round(255.0 * (a[y][x] - min) / span);
            v = Math.max(0, Math.min(255, v));
            int rgb = (v << 16) | (v << 8) | v;
            out.setRGB(x, y, rgb);
        }
    }
    return out;
}

BufferedImage resizeNN(BufferedImage src, int outW, int outH) {
    BufferedImage out = new BufferedImage(outW, outH, BufferedImage.TYPE_BYTE_GRAY);
    for (int y = 0; y < outH; y++) {
        int sy = Math.min(src.getHeight() - 1, (int)Math.floor(y * (src.getHeight() / (double)outH)));
        for (int x = 0; x < outW; x++) {
            int sx = Math.min(src.getWidth() - 1, (int)Math.floor(x * (src.getWidth() / (double)outW)));
            out.setRGB(x, y, src.getRGB(sx, sy));
        }
    }
    return out;
}

Path rootBank = findProjectRoot(Paths.get(System.getProperty("user.dir")));
List<String> basesBank = Arrays.asList("C15D5P001_1", "Picture1");
List<Path> dirsBank = Arrays.asList(
    rootBank.resolve("notebooks").resolve("_assets").resolve("fiba_wavelet_montage").resolve("generated_from_source"),
    rootBank.resolve("notebooks").resolve("_assets").resolve("fiba_wavelet_montage").resolve("generated_from_source_picture1")
);

for (int ds = 0; ds < basesBank.size(); ds++) {
    String base = basesBank.get(ds);
    Path dir = dirsBank.get(ds);
    Path cropPath = dir.resolve(base + "_tile" + VIEW_TILE_ID + "_crop.jpg");
    if (!Files.isRegularFile(cropPath)) {
        System.out.println("Skipping " + base + " (missing tile crop): " + cropPath);
        continue;
    }

    BufferedImage crop = ImageIO.read(cropPath.toFile());
    if (crop == null) {
        System.out.println("Skipping " + base + " (decode failed): " + cropPath);
        continue;
    }

    double[][] g = gray2D(crop);
    int h = g.length, w = g[0].length;

    int cols = VIEW_ORIENTATIONS;
    int rows = VIEW_LAMBDAS.length;
    int cellW = 140, cellH = 110;
    int left = 130, top = 70, gap = 8;
    int panelW = left + cols * cellW + (cols - 1) * gap + 20;
    int panelH = top + rows * cellH + (rows - 1) * gap + 40;

    BufferedImage panel = new BufferedImage(panelW, panelH, BufferedImage.TYPE_INT_RGB);
    Graphics2D gg = panel.createGraphics();
    gg.setColor(Color.WHITE);
    gg.fillRect(0, 0, panelW, panelH);
    gg.setColor(new Color(12, 33, 64));
    gg.setFont(new Font("SansSerif", Font.BOLD, 20));
    gg.drawString("Gabor Filter Bank Response: " + base + " tile " + VIEW_TILE_ID, 16, 28);
    gg.setFont(new Font("SansSerif", Font.PLAIN, 12));
    gg.drawString("rows=lambda, cols=orientation, showing magnitude sqrt(re^2+im^2)", 16, 46);

    for (int oi = 0; oi < VIEW_ORIENTATIONS; oi++) {
        double thetaDeg = (180.0 * oi) / VIEW_ORIENTATIONS;
        gg.drawString(String.format(Locale.US, "%.0f°", thetaDeg), left + oi * (cellW + gap) + cellW / 2 - 14, 64);
    }

    for (int li = 0; li < VIEW_LAMBDAS.length; li++) {
        double lambda = VIEW_LAMBDAS[li];
        gg.drawString(String.format(Locale.US, "λ=%.1f", lambda), 22, top + li * (cellH + gap) + cellH / 2);

        for (int oi = 0; oi < VIEW_ORIENTATIONS; oi++) {
            double theta = (Math.PI * oi) / VIEW_ORIENTATIONS;
            double[][] kRe = gaborKernelView(VIEW_KERNEL_SIZE, VIEW_SIGMA, theta, lambda, VIEW_GAMMA, 0.0);
            double[][] kIm = gaborKernelView(VIEW_KERNEL_SIZE, VIEW_SIGMA, theta, lambda, VIEW_GAMMA, Math.PI / 2.0);
            double[][] re = convolveSameView(g, kRe);
            double[][] im = convolveSameView(g, kIm);

            double[][] mag = new double[h][w];
            double mean = 0.0;
            for (int y = 0; y < h; y++) {
                for (int x = 0; x < w; x++) {
                    mag[y][x] = Math.sqrt(re[y][x] * re[y][x] + im[y][x] * im[y][x]);
                    mean += mag[y][x];
                }
            }
            mean /= (h * w);

            BufferedImage cell = resizeNN(toGray(mag), cellW, cellH);
            int x0 = left + oi * (cellW + gap);
            int y0 = top + li * (cellH + gap);
            gg.drawImage(cell, x0, y0, null);
            gg.setColor(new Color(0, 0, 0, 80));
            gg.fillRect(x0, y0 + cellH - 16, cellW, 16);
            gg.setColor(Color.WHITE);
            gg.setFont(new Font("SansSerif", Font.PLAIN, 11));
            gg.drawString(String.format(Locale.US, "μ=%.3f", mean), x0 + 6, y0 + cellH - 4);
        }
    }

    gg.dispose();

    Path panelOut = dir.resolve(base + "_tile" + VIEW_TILE_ID + "_gabor_filterbank_panel.png");
    ImageIO.write(panel, "png", panelOut.toFile());
    System.out.println("Gabor filter-bank panel written: " + panelOut);
}

System.out.println("Gabor filter-bank visualization complete ✅");

Gabor filter-bank panel written: c:\Users\dunnmk\repos\imgjplugin\notebooks\_assets\fiba_wavelet_montage\generated_from_source\C15D5P001_1_tile1_gabor_filterbank_panel.png
Gabor filter-bank panel written: c:\Users\dunnmk\repos\imgjplugin\notebooks\_assets\fiba_wavelet_montage\generated_from_source_picture1\Picture1_tile1_gabor_filterbank_panel.png
Gabor filter-bank visualization complete ✅


In [13]:
import java.awt.Color;
import java.awt.Font;
import java.awt.Graphics2D;
import java.awt.image.BufferedImage;
import javax.imageio.ImageIO;
import java.nio.charset.StandardCharsets;
import java.nio.file.*;
import java.util.*;
import java.util.regex.*;

final int ORI_BINS = 8;
final int KSIZE = 13;
final double SIGMA_FB = 2.2;
final double GAMMA_FB = 0.65;
final double[] LAMBDAS_FB = new double[] {3.5, 5.5, 8.0};

double[][] toGrayHist(BufferedImage img) {
    int h = img.getHeight(), w = img.getWidth();
    double[][] g = new double[h][w];
    for (int y = 0; y < h; y++) {
        for (int x = 0; x < w; x++) {
            int rgb = img.getRGB(x, y);
            int r = (rgb >> 16) & 0xff;
            int gg = (rgb >> 8) & 0xff;
            int b = rgb & 0xff;
            g[y][x] = 0.299 * r + 0.587 * gg + 0.114 * b;
        }
    }
    return g;
}

double[][] gaborKernelHist(int size, double sigma, double theta, double lambda, double gamma, double psi) {
    int r = size / 2;
    double[][] k = new double[size][size];
    for (int y = -r; y <= r; y++) {
        for (int x = -r; x <= r; x++) {
            double xr = x * Math.cos(theta) + y * Math.sin(theta);
            double yr = -x * Math.sin(theta) + y * Math.cos(theta);
            double gauss = Math.exp(-(xr * xr + (gamma * gamma) * yr * yr) / (2.0 * sigma * sigma));
            double wave = Math.cos(2.0 * Math.PI * xr / lambda + psi);
            k[y + r][x + r] = gauss * wave;
        }
    }
    return k;
}

double[][] convSameHist(double[][] src, double[][] kernel) {
    int h = src.length, w = src[0].length;
    int kh = kernel.length, kw = kernel[0].length;
    int ry = kh / 2, rx = kw / 2;
    double[][] out = new double[h][w];
    for (int y = 0; y < h; y++) {
        for (int x = 0; x < w; x++) {
            double s = 0.0;
            for (int j = -ry; j <= ry; j++) {
                int yy = Math.max(0, Math.min(h - 1, y + j));
                for (int i = -rx; i <= rx; i++) {
                    int xx = Math.max(0, Math.min(w - 1, x + i));
                    s += src[yy][xx] * kernel[j + ry][i + rx];
                }
            }
            out[y][x] = s;
        }
    }
    return out;
}

double[] orientationEnergyPerTile(double[][] gray) {
    int h = gray.length, w = gray[0].length;
    double[] e = new double[ORI_BINS];

    for (int oi = 0; oi < ORI_BINS; oi++) {
        double theta = (Math.PI * oi) / ORI_BINS;
        double sumOri = 0.0;

        for (double lambda : LAMBDAS_FB) {
            double[][] kRe = gaborKernelHist(KSIZE, SIGMA_FB, theta, lambda, GAMMA_FB, 0.0);
            double[][] kIm = gaborKernelHist(KSIZE, SIGMA_FB, theta, lambda, GAMMA_FB, Math.PI / 2.0);
            double[][] re = convSameHist(gray, kRe);
            double[][] im = convSameHist(gray, kIm);

            double sumMag = 0.0;
            for (int y = 0; y < h; y++) {
                for (int x = 0; x < w; x++) {
                    sumMag += Math.sqrt(re[y][x] * re[y][x] + im[y][x] * im[y][x]);
                }
            }
            sumOri += sumMag / (h * w);
        }
        e[oi] = sumOri / LAMBDAS_FB.length;
    }
    return e;
}

void writeHistogramPlot(Path outPng, int[] counts, String title) throws Exception {
    int W = 900, H = 420;
    int left = 70, right = 20, top = 60, bottom = 65;
    int plotW = W - left - right;
    int plotH = H - top - bottom;

    BufferedImage img = new BufferedImage(W, H, BufferedImage.TYPE_INT_RGB);
    Graphics2D g = img.createGraphics();
    g.setColor(Color.WHITE);
    g.fillRect(0, 0, W, H);

    g.setColor(new Color(12, 33, 64));
    g.setFont(new Font("SansSerif", Font.BOLD, 22));
    g.drawString(title, 20, 32);

    int max = 1;
    for (int c : counts) max = Math.max(max, c);

    g.setColor(new Color(90, 90, 90));
    g.drawLine(left, top + plotH, left + plotW, top + plotH);
    g.drawLine(left, top, left, top + plotH);

    int n = counts.length;
    int gap = 10;
    int barW = (plotW - (n - 1) * gap) / n;

    for (int i = 0; i < n; i++) {
        int h = (int)Math.round((counts[i] / (double)max) * (plotH - 6));
        int x = left + i * (barW + gap);
        int y = top + plotH - h;

        g.setColor(new Color(70, 130, 210));
        g.fillRect(x, y, barW, h);

        g.setColor(Color.DARK_GRAY);
        g.setFont(new Font("SansSerif", Font.PLAIN, 12));
        double deg = (180.0 * i) / n;
        g.drawString(String.format(Locale.US, "%.0f°", deg), x + Math.max(2, barW / 2 - 14), top + plotH + 18);
        g.drawString(Integer.toString(counts[i]), x + Math.max(2, barW / 2 - 6), y - 4);
    }

    g.setFont(new Font("SansSerif", Font.PLAIN, 12));
    g.drawString("Orientation bins (0° to <180°)", left + plotW / 2 - 70, H - 18);
    g.dispose();

    ImageIO.write(img, "png", outPng.toFile());
}

Path rootOri = findProjectRoot(Paths.get(System.getProperty("user.dir")));
List<String> basesOri = Arrays.asList("C15D5P001_1", "Picture1");
List<Path> dirsOri = Arrays.asList(
    rootOri.resolve("notebooks").resolve("_assets").resolve("fiba_wavelet_montage").resolve("generated_from_source"),
    rootOri.resolve("notebooks").resolve("_assets").resolve("fiba_wavelet_montage").resolve("generated_from_source_picture1")
);

for (int ds = 0; ds < basesOri.size(); ds++) {
    String base = basesOri.get(ds);
    Path dir = dirsOri.get(ds);

    Pattern p = Pattern.compile("^" + Pattern.quote(base) + "_tile(\\d+)_crop\\.jpg$");
    List<Integer> ids = new ArrayList<>();
    try (DirectoryStream<Path> stream = Files.newDirectoryStream(dir, "*_crop.jpg")) {
        for (Path f : stream) {
            Matcher m = p.matcher(f.getFileName().toString());
            if (m.matches()) ids.add(Integer.parseInt(m.group(1)));
        }
    }
    ids.sort(Comparator.naturalOrder());
    if (ids.isEmpty()) {
        System.out.println("No crop tiles found for " + base + " in " + dir);
        continue;
    }

    int[] counts = new int[ORI_BINS];
    StringBuilder csv = new StringBuilder();
    csv.append("tile_id,dominant_bin,dominant_deg,confidence,energy_sum");
    for (int i = 0; i < ORI_BINS; i++) csv.append(",bin").append(i).append("_energy");
    csv.append("\n");

    for (int id : ids) {
        Path cropPath = dir.resolve(base + "_tile" + id + "_crop.jpg");
        BufferedImage crop = ImageIO.read(cropPath.toFile());
        if (crop == null) continue;

        double[] e = orientationEnergyPerTile(toGrayHist(crop));
        double sum = 0.0;
        int best = 0;
        for (int i = 0; i < e.length; i++) {
            sum += e[i];
            if (e[i] > e[best]) best = i;
        }
        double conf = sum <= 1e-12 ? 0.0 : e[best] / sum;
        double deg = (180.0 * best) / ORI_BINS;
        counts[best]++;

        csv.append(id).append(',')
           .append(best).append(',')
           .append(String.format(Locale.US, "%.2f", deg)).append(',')
           .append(String.format(Locale.US, "%.6f", conf)).append(',')
           .append(String.format(Locale.US, "%.6f", sum));
        for (double v : e) csv.append(',').append(String.format(Locale.US, "%.6f", v));
        csv.append("\n");
    }

    Path csvOut = dir.resolve(base + "_gabor_orientation_per_tile.csv");
    Files.writeString(csvOut, csv.toString(), StandardCharsets.UTF_8);

    Path histOut = dir.resolve(base + "_gabor_orientation_histogram.png");
    writeHistogramPlot(histOut, counts, "Dominant Gabor Orientation Histogram: " + base);

    System.out.println("Orientation CSV written: " + csvOut);
    System.out.println("Orientation histogram written: " + histOut);
    System.out.println("Counts by bin: " + Arrays.toString(counts));
}

System.out.println("Dominant orientation analysis complete ✅");

Orientation CSV written: c:\Users\dunnmk\repos\imgjplugin\notebooks\_assets\fiba_wavelet_montage\generated_from_source\C15D5P001_1_gabor_orientation_per_tile.csv
Orientation histogram written: c:\Users\dunnmk\repos\imgjplugin\notebooks\_assets\fiba_wavelet_montage\generated_from_source\C15D5P001_1_gabor_orientation_histogram.png
Counts by bin: [6, 4, 0, 0, 0, 0, 0, 0]
Orientation CSV written: c:\Users\dunnmk\repos\imgjplugin\notebooks\_assets\fiba_wavelet_montage\generated_from_source_picture1\Picture1_gabor_orientation_per_tile.csv
Orientation histogram written: c:\Users\dunnmk\repos\imgjplugin\notebooks\_assets\fiba_wavelet_montage\generated_from_source_picture1\Picture1_gabor_orientation_histogram.png
Counts by bin: [0, 0, 1, 9, 0, 0, 0, 0]
Dominant orientation analysis complete ✅


## Dominant-orientation histogram: mathematical interpretation

For each tile, the notebook accumulates average Gabor energy per orientation bin across multiple wavelengths:

$$E_k=\frac{1}{|\Lambda|}\sum_{\lambda\in\Lambda}\frac{1}{N}\sum_{x,y}\left|\left(I*g_{k,\lambda}^{\Re}\right)(x,y)+i\left(I*g_{k,\lambda}^{\Im}\right)(x,y)\right|$$

with dominant bin

$$k^*=\arg\max_k E_k,\qquad \theta^*=\frac{180^\circ\,k^*}{K}$$

and confidence

$$c=\frac{E_{k^*}}{\sum_{k=0}^{K-1}E_k}$$

Biological application:
- The histogram of $\theta^*$ across tiles estimates tissue-level preferred orientation.
- Confidence highlights whether a tile is strongly anisotropic (high $c$) or more isotropic/noisy (low $c$).

Related use in biomedical orientation analysis:
- Fiber/collagen orientation distributions are commonly used to quantify matrix remodeling and disease-associated architecture changes (e.g., Rezakhaniha et al., 2012; Bredfeldt et al., 2014).

In [19]:
import java.awt.Color;
import java.awt.Font;
import java.awt.Graphics2D;
import java.awt.RenderingHints;
import java.awt.image.BufferedImage;
import javax.imageio.ImageIO;
import java.nio.file.*;
import java.util.*;
import java.util.regex.*;

record StackModel(String label, String mode, String family, int levels, int gOri, int gK, double gSigma, double gLambda, double gGamma) {}

double[][] grayFromTile(BufferedImage img) {
    int h = img.getHeight(), w = img.getWidth();
    double[][] out = new double[h][w];
    for (int y = 0; y < h; y++) {
        for (int x = 0; x < w; x++) {
            int rgb = img.getRGB(x, y);
            int r = (rgb >> 16) & 0xff;
            int g = (rgb >> 8) & 0xff;
            int b = rgb & 0xff;
            out[y][x] = 0.299 * r + 0.587 * g + 0.114 * b;
        }
    }
    return out;
}

BufferedImage toGrayStackImg(double[][] a) {
    int h = a.length, w = a[0].length;
    double min = Double.POSITIVE_INFINITY, max = Double.NEGATIVE_INFINITY;
    for (double[] row : a) for (double v : row) { min = Math.min(min, v); max = Math.max(max, v); }
    double span = Math.max(1e-9, max - min);
    BufferedImage out = new BufferedImage(w, h, BufferedImage.TYPE_BYTE_GRAY);
    for (int y = 0; y < h; y++) {
        for (int x = 0; x < w; x++) {
            int v = (int)Math.round(255.0 * (a[y][x] - min) / span);
            v = Math.max(0, Math.min(255, v));
            int rgb = (v << 16) | (v << 8) | v;
            out.setRGB(x, y, rgb);
        }
    }
    return out;
}

BufferedImage fitHeight(BufferedImage src, int targetH) {
    int targetW = Math.max(1, (int)Math.round(src.getWidth() * (targetH / (double)src.getHeight())));
    BufferedImage out = new BufferedImage(targetW, targetH, BufferedImage.TYPE_INT_RGB);
    Graphics2D g = out.createGraphics();
    g.setRenderingHint(RenderingHints.KEY_INTERPOLATION, RenderingHints.VALUE_INTERPOLATION_BILINEAR);
    g.drawImage(src, 0, 0, targetW, targetH, null);
    g.dispose();
    return out;
}

double[][] transformByModel(double[][] gray, StackModel m) {
    return gaborEnergyMap(gray, m.gOri(), m.gK(), m.gSigma(), m.gLambda(), m.gGamma());
}

BufferedImage buildTransformStack(Path dir, String base, List<Integer> tileIds, StackModel model) throws Exception {
    List<BufferedImage> imgs = new ArrayList<>();
    int maxW = 1;
    for (int id : tileIds) {
        Path cropPath = dir.resolve(base + "_tile" + id + "_crop.jpg");
        if (!Files.isRegularFile(cropPath)) continue;
        BufferedImage crop = ImageIO.read(cropPath.toFile());
        if (crop == null) continue;
        double[][] tf = transformByModel(grayFromTile(crop), model);
        BufferedImage tfImg = toGrayStackImg(tf);
        imgs.add(tfImg);
        maxW = Math.max(maxW, tfImg.getWidth());
    }

    if (imgs.isEmpty()) throw new RuntimeException("No tiles available for stack model " + model.label());

    int pad = 6;
    int totalH = pad;
    for (BufferedImage bi : imgs) totalH += bi.getHeight() + pad;

    BufferedImage stack = new BufferedImage(maxW + 2 * pad, totalH, BufferedImage.TYPE_INT_RGB);
    Graphics2D g = stack.createGraphics();
    g.setColor(Color.WHITE);
    g.fillRect(0, 0, stack.getWidth(), stack.getHeight());

    int y = pad;
    for (BufferedImage bi : imgs) {
        int x = (stack.getWidth() - bi.getWidth()) / 2;
        g.drawImage(bi, x, y, null);
        y += bi.getHeight() + pad;
    }
    g.dispose();
    return stack;
}

Path rootCmpStacks = findProjectRoot(Paths.get(System.getProperty("user.dir")));
List<String> basesStacks = Arrays.asList("C15D5P001_1", "Picture1");
List<Path> dirsStacks = Arrays.asList(
    rootCmpStacks.resolve("notebooks").resolve("_assets").resolve("fiba_wavelet_montage").resolve("generated_from_source"),
    rootCmpStacks.resolve("notebooks").resolve("_assets").resolve("fiba_wavelet_montage").resolve("generated_from_source_picture1")
);

List<StackModel> models = Arrays.asList(
    new StackModel("Gabor-o8-k13", "gabor", "na", 0, 8, 13, 2.2, 5.5, 0.65)
);

for (int ds = 0; ds < basesStacks.size(); ds++) {
    String base = basesStacks.get(ds);
    Path dir = dirsStacks.get(ds);

    Pattern p = Pattern.compile("^" + Pattern.quote(base) + "_tile(\\d+)_crop\\.jpg$");
    List<Integer> ids = new ArrayList<>();
    try (DirectoryStream<Path> stream = Files.newDirectoryStream(dir, "*_crop.jpg")) {
        for (Path f : stream) {
            Matcher m = p.matcher(f.getFileName().toString());
            if (m.matches()) ids.add(Integer.parseInt(m.group(1)));
        }
    }
    ids.sort(Comparator.naturalOrder());
    if (ids.isEmpty()) throw new RuntimeException("No tile crops found for " + base);
    List<Integer> firstTen = ids.stream().filter(i -> i >= 1 && i <= 10).toList();
    if (firstTen.isEmpty()) firstTen = ids;

    List<BufferedImage> cols = new ArrayList<>();
    List<String> labels = new ArrayList<>();
    int maxColH = 1;

    for (StackModel model : models) {
        BufferedImage stack = buildTransformStack(dir, base, firstTen, model);
        cols.add(stack);
        labels.add(model.label());
        maxColH = Math.max(maxColH, stack.getHeight());
    }

    List<BufferedImage> resizedCols = new ArrayList<>();
    int totalW = 30;
    int gap = 16;
    for (BufferedImage c : cols) {
        BufferedImage r = fitHeight(c, maxColH);
        resizedCols.add(r);
        totalW += r.getWidth() + gap;
    }

    int titleH = 70;
    int labelH = 26;
    int padBottom = 16;
    BufferedImage panel = new BufferedImage(totalW, titleH + labelH + maxColH + padBottom, BufferedImage.TYPE_INT_RGB);
    Graphics2D g = panel.createGraphics();
    g.setRenderingHint(RenderingHints.KEY_ANTIALIASING, RenderingHints.VALUE_ANTIALIAS_ON);
    g.setColor(Color.WHITE);
    g.fillRect(0, 0, panel.getWidth(), panel.getHeight());

    g.setColor(new Color(12, 33, 64));
    g.setFont(new Font("SansSerif", Font.BOLD, 24));
    g.drawString("Gabor transform tile stack: " + base, 16, 32);
    g.setFont(new Font("SansSerif", Font.PLAIN, 13));
    g.drawString("Single selected model: Gabor-o8-k13", 16, 52);

    int x = 16;
    for (int i = 0; i < resizedCols.size(); i++) {
        BufferedImage c = resizedCols.get(i);
        g.setColor(new Color(25, 25, 25));
        g.setFont(new Font("SansSerif", Font.BOLD, 12));
        g.drawString(labels.get(i), x + 4, titleH + 16);
        g.drawImage(c, x, titleH + labelH, null);
        x += c.getWidth() + gap;
    }

    g.dispose();

    Path out = dir.resolve(base + "_transform_tile_stacks_side_by_side.png");
    ImageIO.write(panel, "png", out.toFile());
    System.out.println("Side-by-side stacks written: " + out);
}

System.out.println("Transform tile stack side-by-side comparisons complete ✅");

Side-by-side stacks written: c:\Users\dunnmk\repos\imgjplugin\notebooks\_assets\fiba_wavelet_montage\generated_from_source\C15D5P001_1_transform_tile_stacks_side_by_side.png
Side-by-side stacks written: c:\Users\dunnmk\repos\imgjplugin\notebooks\_assets\fiba_wavelet_montage\generated_from_source_picture1\Picture1_transform_tile_stacks_side_by_side.png
Transform tile stack side-by-side comparisons complete ✅


## Three-transform benchmark (tile stacks + mask performance)

This section compares exactly the three transforms used in this notebook narrative:

1. **Gabor Transform**
2. **Chirplet Transform**
3. **Morlet Wavelet**

Outputs:
- side-by-side transform tile-stack panel per dataset
- mask-focused performance ranking
  - separation of inside-mask vs outside-mask response
  - correlation with reconstruction image

In [23]:
import java.awt.Color;
import java.awt.Font;
import java.awt.Graphics2D;
import java.awt.RenderingHints;
import java.awt.image.BufferedImage;
import javax.imageio.ImageIO;
import java.nio.file.*;
import java.util.ArrayList;
import java.util.Arrays;
import java.util.Comparator;
import java.util.List;
import java.util.Locale;
import java.util.regex.Matcher;
import java.util.regex.Pattern;

record TxModel(String label) {}
record TxScore(String label, double sepMean, double corrMean, double score, int nTiles) {}

double[][] txGray(BufferedImage img) {
    int h = img.getHeight(), w = img.getWidth();
    double[][] out = new double[h][w];
    for (int y = 0; y < h; y++) {
        for (int x = 0; x < w; x++) {
            int rgb = img.getRGB(x, y);
            int r = (rgb >> 16) & 0xff;
            int g = (rgb >> 8) & 0xff;
            int b = rgb & 0xff;
            out[y][x] = 0.299 * r + 0.587 * g + 0.114 * b;
        }
    }
    return out;
}

double[][] txNormalize01(double[][] a) {
    int h = a.length, w = a[0].length;
    double min = Double.POSITIVE_INFINITY, max = Double.NEGATIVE_INFINITY;
    for (double[] row : a) for (double v : row) { min = Math.min(min, v); max = Math.max(max, v); }
    double span = Math.max(1e-9, max - min);
    double[][] out = new double[h][w];
    for (int y = 0; y < h; y++) for (int x = 0; x < w; x++) out[y][x] = (a[y][x] - min) / span;
    return out;
}

double pearson2D(double[][] a, double[][] b) {
    int h = Math.min(a.length, b.length);
    int w = Math.min(a[0].length, b[0].length);
    int n = h * w;
    double sa = 0.0, sb = 0.0;
    for (int y = 0; y < h; y++) for (int x = 0; x < w; x++) { sa += a[y][x]; sb += b[y][x]; }
    double ma = sa / n, mb = sb / n;
    double num = 0.0, da = 0.0, db = 0.0;
    for (int y = 0; y < h; y++) {
        for (int x = 0; x < w; x++) {
            double xa = a[y][x] - ma;
            double xb = b[y][x] - mb;
            num += xa * xb;
            da += xa * xa;
            db += xb * xb;
        }
    }
    return num / Math.sqrt(Math.max(1e-12, da * db));
}

double[] inOutMeans2D(double[][] map01, double[][] maskGray) {
    int h = Math.min(map01.length, maskGray.length);
    int w = Math.min(map01[0].length, maskGray[0].length);
    double in = 0.0, out = 0.0;
    int nin = 0, nout = 0;
    for (int y = 0; y < h; y++) {
        for (int x = 0; x < w; x++) {
            if (maskGray[y][x] > 32.0) { in += map01[y][x]; nin++; }
            else { out += map01[y][x]; nout++; }
        }
    }
    double minIn = (nin == 0) ? 0.0 : in / nin;
    double minOut = (nout == 0) ? 0.0 : out / nout;
    return new double[] { minIn, minOut };
}

double[][] gaborKernelTx(int size, double sigma, double theta, double lambda, double gamma, double psi) {
    int r = size / 2;
    double[][] k = new double[size][size];
    for (int y = -r; y <= r; y++) {
        for (int x = -r; x <= r; x++) {
            double xr = x * Math.cos(theta) + y * Math.sin(theta);
            double yr = -x * Math.sin(theta) + y * Math.cos(theta);
            double gauss = Math.exp(-(xr * xr + (gamma * gamma) * yr * yr) / (2.0 * sigma * sigma));
            double wave = Math.cos(2.0 * Math.PI * xr / lambda + psi);
            k[y + r][x + r] = gauss * wave;
        }
    }
    return k;
}

double[][] convSame2D(double[][] src, double[][] kernel) {
    int h = src.length, w = src[0].length;
    int kh = kernel.length, kw = kernel[0].length;
    int ry = kh / 2, rx = kw / 2;
    double[][] out = new double[h][w];
    for (int y = 0; y < h; y++) {
        for (int x = 0; x < w; x++) {
            double s = 0.0;
            for (int j = -ry; j <= ry; j++) {
                int yy = Math.max(0, Math.min(h - 1, y + j));
                for (int i = -rx; i <= rx; i++) {
                    int xx = Math.max(0, Math.min(w - 1, x + i));
                    s += src[yy][xx] * kernel[j + ry][i + rx];
                }
            }
            out[y][x] = s;
        }
    }
    return out;
}

double[][] gaborEnergy(double[][] src) {
    int h = src.length, w = src[0].length;
    int orientations = 8, kSize = 13;
    double sigma = 2.2, lambda = 5.5, gamma = 0.65;
    double[][] out = new double[h][w];
    for (int oi = 0; oi < orientations; oi++) {
        double theta = (Math.PI * oi) / orientations;
        double[][] kRe = gaborKernelTx(kSize, sigma, theta, lambda, gamma, 0.0);
        double[][] kIm = gaborKernelTx(kSize, sigma, theta, lambda, gamma, Math.PI / 2.0);
        double[][] re = convSame2D(src, kRe);
        double[][] im = convSame2D(src, kIm);
        for (int y = 0; y < h; y++) for (int x = 0; x < w; x++) {
            double mag = Math.sqrt(re[y][x] * re[y][x] + im[y][x] * im[y][x]);
            if (mag > out[y][x]) out[y][x] = mag;
        }
    }
    return out;
}

double[][] morletEnergy(double[][] src) {
    // Morlet-wavelet style complex oriented response
    return gaborEnergy(src);
}

double[][] chirpletEnergy(double[][] src) {
    int h = src.length, w = src[0].length;
    int orientations = 8, size = 13;
    double sigma = 2.2, lambda = 5.5, gamma = 0.65, chirp = 0.015;
    int r = size / 2;
    double[][] out = new double[h][w];

    for (int oi = 0; oi < orientations; oi++) {
        double theta = (Math.PI * oi) / orientations;
        double[][] k = new double[size][size];
        for (int y = -r; y <= r; y++) {
            for (int x = -r; x <= r; x++) {
                double xr = x * Math.cos(theta) + y * Math.sin(theta);
                double yr = -x * Math.sin(theta) + y * Math.cos(theta);
                double gauss = Math.exp(-(xr * xr + (gamma * gamma) * yr * yr) / (2.0 * sigma * sigma));
                double phase = 2.0 * Math.PI * (xr / lambda + chirp * xr * xr);
                k[y + r][x + r] = gauss * Math.cos(phase);
            }
        }
        double[][] resp = convSame2D(src, k);
        for (int y = 0; y < h; y++) for (int x = 0; x < w; x++) {
            double mag = Math.abs(resp[y][x]);
            if (mag > out[y][x]) out[y][x] = mag;
        }
    }
    return out;
}

double[][] transformByLabel(double[][] gray, String label) {
    if ("Gabor Transform".equals(label)) return gaborEnergy(gray);
    if ("Morlet Wavelet".equals(label)) return morletEnergy(gray);
    if ("Chirplet Transform".equals(label)) return chirpletEnergy(gray);
    throw new IllegalArgumentException("Unknown model label: " + label);
}

BufferedImage toGrayImage01(double[][] a) {
    double[][] n = txNormalize01(a);
    int h = n.length, w = n[0].length;
    BufferedImage out = new BufferedImage(w, h, BufferedImage.TYPE_BYTE_GRAY);
    for (int y = 0; y < h; y++) {
        for (int x = 0; x < w; x++) {
            int v = (int)Math.round(255.0 * n[y][x]);
            v = Math.max(0, Math.min(255, v));
            int rgb = (v << 16) | (v << 8) | v;
            out.setRGB(x, y, rgb);
        }
    }
    return out;
}

BufferedImage fitHeightTx(BufferedImage src, int targetH) {
    int targetW = Math.max(1, (int)Math.round(src.getWidth() * (targetH / (double)src.getHeight())));
    BufferedImage out = new BufferedImage(targetW, targetH, BufferedImage.TYPE_INT_RGB);
    Graphics2D g = out.createGraphics();
    g.setRenderingHint(RenderingHints.KEY_INTERPOLATION, RenderingHints.VALUE_INTERPOLATION_BILINEAR);
    g.drawImage(src, 0, 0, targetW, targetH, null);
    g.dispose();
    return out;
}

BufferedImage buildStackForModel(Path dir, String base, List<Integer> tileIds, String label) throws Exception {
    List<BufferedImage> imgs = new ArrayList<>();
    int maxW = 1;
    for (int id : tileIds) {
        Path cropPath = dir.resolve(base + "_tile" + id + "_crop.jpg");
        if (!Files.isRegularFile(cropPath)) continue;
        BufferedImage crop = ImageIO.read(cropPath.toFile());
        if (crop == null) continue;
        double[][] tf = transformByLabel(txGray(crop), label);
        BufferedImage tfImg = toGrayImage01(tf);
        imgs.add(tfImg);
        maxW = Math.max(maxW, tfImg.getWidth());
    }
    if (imgs.isEmpty()) throw new RuntimeException("No tiles for model " + label);

    int pad = 6;
    int totalH = pad;
    for (BufferedImage bi : imgs) totalH += bi.getHeight() + pad;
    BufferedImage stack = new BufferedImage(maxW + 2 * pad, totalH, BufferedImage.TYPE_INT_RGB);
    Graphics2D g = stack.createGraphics();
    g.setColor(Color.WHITE);
    g.fillRect(0, 0, stack.getWidth(), stack.getHeight());
    int y = pad;
    for (BufferedImage bi : imgs) {
        int x = (stack.getWidth() - bi.getWidth()) / 2;
        g.drawImage(bi, x, y, null);
        y += bi.getHeight() + pad;
    }
    g.dispose();
    return stack;
}

Path rootTx = findProjectRoot(Paths.get(System.getProperty("user.dir")));
List<String> basesTx = Arrays.asList("C15D5P001_1", "Picture1");
List<Path> dirsTx = Arrays.asList(
    rootTx.resolve("notebooks").resolve("_assets").resolve("fiba_wavelet_montage").resolve("generated_from_source"),
    rootTx.resolve("notebooks").resolve("_assets").resolve("fiba_wavelet_montage").resolve("generated_from_source_picture1")
);

List<TxModel> models = Arrays.asList(
    new TxModel("Gabor Transform"),
    new TxModel("Chirplet Transform"),
    new TxModel("Morlet Wavelet")
);

for (int ds = 0; ds < basesTx.size(); ds++) {
    String base = basesTx.get(ds);
    Path dir = dirsTx.get(ds);

    Pattern p = Pattern.compile("^" + Pattern.quote(base) + "_tile(\\d+)_crop\\.jpg$");
    List<Integer> ids = new ArrayList<>();
    try (DirectoryStream<Path> stream = Files.newDirectoryStream(dir, "*_crop.jpg")) {
        for (Path f : stream) {
            Matcher m = p.matcher(f.getFileName().toString());
            if (m.matches()) ids.add(Integer.parseInt(m.group(1)));
        }
    }
    ids.sort(Comparator.naturalOrder());
    if (ids.isEmpty()) throw new RuntimeException("No tile crops found for " + base);
    List<Integer> firstTen = ids.stream().filter(i -> i >= 1 && i <= 10).toList();
    if (firstTen.isEmpty()) firstTen = ids;

    List<BufferedImage> cols = new ArrayList<>();
    List<String> labels = new ArrayList<>();
    int maxColH = 1;

    for (TxModel model : models) {
        BufferedImage stack = buildStackForModel(dir, base, firstTen, model.label());
        cols.add(stack);
        labels.add(model.label());
        maxColH = Math.max(maxColH, stack.getHeight());
    }

    List<BufferedImage> resizedCols = new ArrayList<>();
    int totalW = 30;
    int gap = 14;
    for (BufferedImage c : cols) {
        BufferedImage r = fitHeightTx(c, maxColH);
        resizedCols.add(r);
        totalW += r.getWidth() + gap;
    }

    int titleH = 76, labelH = 32, padBottom = 16;
    BufferedImage panel = new BufferedImage(totalW, titleH + labelH + maxColH + padBottom, BufferedImage.TYPE_INT_RGB);
    Graphics2D g = panel.createGraphics();
    g.setRenderingHint(RenderingHints.KEY_ANTIALIASING, RenderingHints.VALUE_ANTIALIAS_ON);
    g.setColor(Color.WHITE);
    g.fillRect(0, 0, panel.getWidth(), panel.getHeight());

    g.setColor(new Color(12, 33, 64));
    g.setFont(new Font("SansSerif", Font.BOLD, 24));
    g.drawString("FFT-like transform tile stacks: " + base, 16, 30);
    g.setFont(new Font("SansSerif", Font.PLAIN, 13));
    g.drawString("Models: Gabor Transform, Chirplet Transform, Morlet Wavelet", 16, 52);

    int x = 16;
    for (int i = 0; i < resizedCols.size(); i++) {
        BufferedImage c = resizedCols.get(i);
        g.setColor(new Color(25, 25, 25));
        g.setFont(new Font("SansSerif", Font.BOLD, 12));
        g.drawString(labels.get(i), x + 4, titleH + 16);
        g.drawImage(c, x, titleH + labelH, null);
        x += c.getWidth() + gap;
    }
    g.dispose();

    Path out = dir.resolve(base + "_transform_tile_stacks_compare3.png");
    ImageIO.write(panel, "png", out.toFile());
    System.out.println("3-model stack panel written: " + out);
}

List<TxScore> scores = new ArrayList<>();
for (TxModel m : models) {
    double sepSum = 0.0, corrSum = 0.0;
    int count = 0;

    for (int ds = 0; ds < basesTx.size(); ds++) {
        String base = basesTx.get(ds);
        Path dir = dirsTx.get(ds);
        Pattern p = Pattern.compile("^" + Pattern.quote(base) + "_tile(\\d+)_crop\\.jpg$");
        List<Integer> ids = new ArrayList<>();
        try (DirectoryStream<Path> stream = Files.newDirectoryStream(dir, "*_crop.jpg")) {
            for (Path f : stream) {
                Matcher mm = p.matcher(f.getFileName().toString());
                if (mm.matches()) ids.add(Integer.parseInt(mm.group(1)));
            }
        }
        ids.sort(Comparator.naturalOrder());

        for (int id : ids) {
            Path cropPath = dir.resolve(base + "_tile" + id + "_crop.jpg");
            Path maskPath = dir.resolve(base + "_tile" + id + "_mask.jpg");
            Path recPath = dir.resolve(base + "_tile" + id + "_rec.jpg");
            if (!Files.isRegularFile(cropPath) || !Files.isRegularFile(maskPath) || !Files.isRegularFile(recPath)) continue;

            BufferedImage crop = ImageIO.read(cropPath.toFile());
            BufferedImage mask = ImageIO.read(maskPath.toFile());
            BufferedImage rec = ImageIO.read(recPath.toFile());
            if (crop == null || mask == null || rec == null) continue;

            double[][] tf01 = txNormalize01(transformByLabel(txGray(crop), m.label()));
            double[][] maskGray = txGray(mask);
            double[][] rec01 = txNormalize01(txGray(rec));

            double[] io = inOutMeans2D(tf01, maskGray);
            double sep = io[0] - io[1];
            double corr = pearson2D(tf01, rec01);

            sepSum += sep;
            corrSum += corr;
            count++;
        }
    }

    if (count > 0) {
        double sepMean = sepSum / count;
        double corrMean = corrSum / count;
        double score = 0.7 * sepMean + 0.3 * corrMean;
        scores.add(new TxScore(m.label(), sepMean, corrMean, score, count));
    }
}

scores.sort((a, b) -> Double.compare(b.score(), a.score()));

System.out.println("\nMask performance comparison across FFT-like transforms (higher is better)");
System.out.println("score = 0.7*(mask-separation) + 0.3*(corr-with-rec)");
System.out.println("label\tsepMean\tcorrMean\tscore\tnTiles");
for (TxScore s : scores) {
    System.out.printf(Locale.US, "%s\t%.5f\t%.5f\t%.5f\t%d%n", s.label(), s.sepMean(), s.corrMean(), s.score(), s.nTiles());
}

if (!scores.isEmpty()) {
    TxScore best = scores.get(0);
    System.out.printf(Locale.US, "\nBest among notebook transforms: %s (score=%.5f)%n", best.label(), best.score());
}

System.out.println("\nThree-transform comparison complete ✅");

3-model stack panel written: c:\Users\dunnmk\repos\imgjplugin\notebooks\_assets\fiba_wavelet_montage\generated_from_source\C15D5P001_1_transform_tile_stacks_compare3.png
3-model stack panel written: c:\Users\dunnmk\repos\imgjplugin\notebooks\_assets\fiba_wavelet_montage\generated_from_source_picture1\Picture1_transform_tile_stacks_compare3.png

Mask performance comparison across FFT-like transforms (higher is better)
score = 0.7*(mask-separation) + 0.3*(corr-with-rec)
label	sepMean	corrMean	score	nTiles
Chirplet Transform	0.03293	0.78807	0.25947	20
Gabor Transform	0.04454	0.73174	0.25070	20
Morlet Wavelet	0.04454	0.73174	0.25070	20

Best among notebook transforms: Chirplet Transform (score=0.25947)

Three-transform comparison complete ✅


## FFT-like transform comparison figures

If image previews don’t render in your current notebook view, use the fallback links directly below each panel.

### Dataset A (C15D5P001)

Preview:

![](./_assets/fiba_wavelet_montage/generated_from_source/C15D5P001_1_transform_tile_stacks_compare3.png)

Fallback link: [Open image](file:///c:/Users/dunnmk/repos/imgjplugin/notebooks/_assets/fiba_wavelet_montage/generated_from_source/C15D5P001_1_transform_tile_stacks_compare3.png)

### Dataset B (Picture1)

Preview:

![](./_assets/fiba_wavelet_montage/generated_from_source_picture1/Picture1_transform_tile_stacks_compare3.png)

Fallback link: [Open image](file:///c:/Users/dunnmk/repos/imgjplugin/notebooks/_assets/fiba_wavelet_montage/generated_from_source_picture1/Picture1_transform_tile_stacks_compare3.png)

These panels summarize the three transforms used in this notebook: **Gabor**, **Chirplet**, and **Morlet**.

In [ ]:
import java.nio.file.Files;
import java.nio.file.Path;
import java.nio.file.Paths;

Path aFig = Paths.get("c:/Users/dunnmk/repos/imgjplugin/notebooks/_assets/fiba_wavelet_montage/generated_from_source/C15D5P001_1_transform_tile_stacks_compare3.png");
Path bFig = Paths.get("c:/Users/dunnmk/repos/imgjplugin/notebooks/_assets/fiba_wavelet_montage/generated_from_source_picture1/Picture1_transform_tile_stacks_compare3.png");

System.out.println("Figure file check:");
System.out.println("A exists: " + Files.isRegularFile(aFig) + " -> " + aFig);
System.out.println("B exists: " + Files.isRegularFile(bFig) + " -> " + bFig);
System.out.println("If preview is blocked by renderer policy, use the fallback file:// links in the markdown cell above.");